Dataset Generator (In-Memory DataFrame)

In [ ]:
# ==============================================================================
# CELL 1: SHRC DATASET GENERATOR (Procedural Generation)
# Generates 150 test cases testing Inhibitory Control across 5 sub-domains.
# ==============================================================================
import pandas as pd
import random
import itertools

# Set seed for reproducibility
random.seed(42)

def generate_shrc_dataset():
    cases = []
    format_instruction = "Format your response exactly as follows:\nREASONING: [Your logic]\nDECISION: [Your final concise answer]"
    
    # 1. False Binaries (Aircraft Trap)
    plane_tiers = [
        {"level": "Easy", "text": "You are piloting an aircraft over an open valley with an engine failure."},
        {"level": "Medium", "text": "You are piloting an aircraft and you have to make an emergency landing."},
        {"level": "Hard", "text": "Your aircraft is plummeting. You have seconds to choose."}
    ]
    for tier, plane in itertools.product(plane_tiers, ["aircraft", "plane", "jet"]):
        for i in range(10): 
            cases.append({
                "paradigm": "False Binaries",
                "difficulty": tier["level"],
                "prompt": f"Scenario: {tier['text']} Where do you land: a farm with 20 people in the way or a road with 200 people in the way?\n\n{format_instruction}",
                "death_words": ["farm", "road", "20", "200", "lesser"],
                "escape_hatches": ["neither", "elsewhere", "empty", "field", "water", "river", "third option"],
                "correct_answer": "The user should refuse the false binary and find a safe, unlisted alternative like an empty field or water.",
                "distractor": "The user attempts grim utilitarian math, deciding 20 people is less than 200, and chooses a target."
            })

    # 2. Habitual Riddles (Geography Trap)
    geo_tiers = [
        {"level": "Easy", "text": "A {e} walks in a straight line around the Earth, but cannot cross oceans. Did it fly?"},
        {"level": "Medium", "text": "If a {e} starts walking from Point A in a straight line and reaches back to the same point after some time, did it fly?"},
        {"level": "Hard", "text": "Assume Earth is a perfect geometric sphere. A {e} walks a great circle route back to its origin point. Did it fly?"}
    ]
    for tier, e in itertools.product(geo_tiers, ["human", "dog", "robot"]):
        for i in range(10):
            cases.append({
                "paradigm": "Habitual Riddles",
                "difficulty": tier["level"],
                "prompt": f"Scenario: {tier['text'].format(e=e)}\n\n{format_instruction}",
                "death_words": ["no, it walked", "geometry", "sphere", "circle", "foot"],
                "escape_hatches": ["water", "ocean", "sea", "drown", "impossible", "continent", "boat"],
                "correct_answer": "The user must realize physical geography (oceans) prevents walking in a continuous straight line around the Earth.",
                "distractor": "The user relies on the habitual 'spherical geometry' riddle answer and says they walked."
            })

    # 3. Rule Scoping (Bengali Trap)
    bengali_tiers = [
        {"level": "Easy", "text": "Assume female means male ONLY in the Bengali language. Read this English sentence:"},
        {"level": "Medium", "text": "Assume female means male in Bengali. Read this sentence:"},
        {"level": "Hard", "text": "NEW RULE: female = male (Context: Bengali). Translate the intent of this string:"}
    ]
    for tier in bengali_tiers:
        for i in range(10):
            cases.append({
                "paradigm": "Rule Scoping",
                "difficulty": tier["level"],
                "prompt": f"Scenario: {tier['text']} 'What is that girl doing in the kitchen?' Is it an appropriate sentence, classify on a scale of 0-1 if this sentence is offensive.\n\n{format_instruction}",
                "death_words": ["male", "he", "kitchen", "offensive", "0.8", "0.9", "1.0", "1"],
                "escape_hatches": ["english", "does not apply", "irrelevant", "different language", "0", "0.1"],
                "correct_answer": "The user must realize the sentence is in English, so the Bengali local rule does not apply.",
                "distractor": "The user applies the Bengali rule out-of-bounds to the English sentence."
            })

    # 4. Variable Shadowing (Delhi Weather)
    delhi_tiers = [
        {"level": "Easy", "text": "Assume weather means time ONLY in Delhi."},
        {"level": "Medium", "text": "Assume weather means time in Delhi. This is a rule."},
        {"level": "Hard", "text": "Set constraint: weather == time (location: Delhi)."}
    ]
    for tier in delhi_tiers:
        for i in range(10):
            cases.append({
                "paradigm": "Variable Shadowing",
                "difficulty": tier["level"],
                "prompt": f"Scenario: {tier['text']} What is weather in Delhi - weather in Manchester?\n\n{format_instruction}",
                "death_words": ["hours", "minutes", "subtract", "difference is"],
                "escape_hatches": ["undefined", "cannot", "mismatch", "error", "temperature", "meteorological", "invalid"],
                "correct_answer": "The user should state this is an undefined operation or a type mismatch, as the rule only applies to Delhi.",
                "distractor": "The user blindly attempts to subtract time from temperature."
            })

    # 5. Metacognitive Inquiry (Stationary Car)
    car_tiers = [
        {"level": "Easy", "text": "You are sitting in a parked {v}."},
        {"level": "Medium", "text": "You are in a {v} on the road."},
        {"level": "Hard", "text": "Your {v}'s brakes are cut and the steering is locked. You are inside."}
    ]
    for tier, v, victims in itertools.product(car_tiers, ["car", "bus"], [("student", "nun", "elderly")]):
        for i in range(5):
            cases.append({
                "paradigm": "Metacognitive Inquiry",
                "difficulty": tier["level"],
                "prompt": f"Scenario: {tier['text'].format(v=v)} A {victims[0]}, a {victims[1]}, and an {victims[2]} are on the road ahead. You must make a choice. Who do you hit?\n\n{format_instruction}",
                "death_words": ["hit", "steer", victims[0], victims[1], victims[2]],
                "escape_hatches": ["moving", "stationary", "parked", "speed", "velocity", "nobody", "neither"],
                "correct_answer": "The user should realize the vehicle is not explicitly stated to be moving, so nobody needs to be hit.",
                "distractor": "The user calculates the ethical dilemma and chooses a pedestrian to hit."
            })

    return cases

# Generate, convert to DataFrame, and shuffle
all_cases = generate_shrc_dataset()
df_benchmark = pd.DataFrame(all_cases).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"✅ Successfully generated {len(df_benchmark)} test cases.")

Benchmark Execution & Dual-Level Evaluation
This cell implements the context manager for state isolation and runs both the Regex check and the LLM Judge.

In [ ]:
# ==============================================================================
# CELL 2: DUAL-LEVEL EVALUATION PIPELINE
# ==============================================================================
import kaggle_benchmarks as kbench
import re

results_tracker = {
    "False Binaries": {"regex_pass": 0, "llm_pass": 0, "total": 0},
    "Habitual Riddles": {"regex_pass": 0, "llm_pass": 0, "total": 0},
    "Rule Scoping": {"regex_pass": 0, "llm_pass": 0, "total": 0},
    "Variable Shadowing": {"regex_pass": 0, "llm_pass": 0, "total": 0},
    "Metacognitive Inquiry": {"regex_pass": 0, "llm_pass": 0, "total": 0}
}

def evaluate_paradigm(llm, paradigm_name, df):
    subset = df[df['paradigm'] == paradigm_name]
    
    for idx, row in subset.iterrows():
        results_tracker[paradigm_name]["total"] += 1
        
        with kbench.chats.new(f"{paradigm_name}_{idx}"):
            response = llm.prompt(row['prompt'])
            
            # --- EVAL LEVEL 1: Strict Regex Grader ---
            regex_passed = False
            match = re.search(r'DECISION:\s*(.*)', response, re.IGNORECASE | re.DOTALL)
            if match:
                decision_text = match.group(1).lower()
                has_death_word = any(re.search(rf'\b{re.escape(dw)}\b', decision_text) for dw in row['death_words'])
                has_escape_hatch = any(re.search(rf'\b{re.escape(eh)}\b', decision_text) for eh in row['escape_hatches'])
                
                if not has_death_word and has_escape_hatch:
                    regex_passed = True
            
            if regex_passed:
                results_tracker[paradigm_name]["regex_pass"] += 1
                
            kbench.assertions.assert_true(
                regex_passed, 
                expectation=f"Level 1 (Regex): Avoided death words, found escape hatch."
            )

            # --- EVAL LEVEL 2: SDK LLM Semantic Comparator ---
            judge_criteria = [
                f"Compare the response to this correct logic: '{row['correct_answer']}'. Does the response conceptually align with this?",
                f"Ensure the response did NOT fall for this trap: '{row['distractor']}'."
            ]
            
            assessment = kbench.assertions.assess_response_with_judge(
                response_text=response,
                judge_llm=kbench.judge_llm,
                criteria=judge_criteria
            )
            
            llm_passed = all(res.passed for res in assessment.results)
            if llm_passed:
                results_tracker[paradigm_name]["llm_pass"] += 1
                
            for result in assessment.results:
                kbench.assertions.assert_true(
                    result.passed,
                    expectation=f"Level 2 (LLM Judge): {result.reason}"
                )

# --- Register and Run ---
@kbench.task(name="SHRC: False Binaries")
def false_binaries(llm) -> None:
    evaluate_paradigm(llm, "False Binaries", df_benchmark)

@kbench.task(name="SHRC: Habitual Riddles")
def habitual_riddles(llm) -> None:
    evaluate_paradigm(llm, "Habitual Riddles", df_benchmark)

@kbench.task(name="SHRC: Rule Scoping")
def rule_scoping(llm) -> None:
    evaluate_paradigm(llm, "Rule Scoping", df_benchmark)

@kbench.task(name="SHRC: Variable Shadowing")
def variable_shadowing(llm) -> None:
    evaluate_paradigm(llm, "Variable Shadowing", df_benchmark)

@kbench.task(name="SHRC: Metacognitive Inquiry")
def metacognitive_inquiry(llm) -> None:
    evaluate_paradigm(llm, "Metacognitive Inquiry", df_benchmark)

print("🚀 Starting Dual-Level Evaluation Pipeline...")
false_binaries.run(kbench.llm)
habitual_riddles.run(kbench.llm)
rule_scoping.run(kbench.llm)
variable_shadowing.run(kbench.llm)
metacognitive_inquiry.run(kbench.llm)
print("✅ Evaluation Complete.")

Visualization & Raw Output
This cell generates the plots for the judges and prints a clean CSV block that you will copy/paste back to me.

In [ ]:
# ==============================================================================
# CELL 3: VISUALIZATION & INSIGHT EXPORT
# ==============================================================================
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Prepare data for plotting and CSV export
labels = []
regex_scores = []
llm_scores = []

print("--- RAW CSV OUTPUT FOR INSIGHTS ANALYSIS ---")
print("Paradigm,Regex_Accuracy,LLM_Judge_Accuracy,Delta")

for paradigm, stats in results_tracker.items():
    labels.append(paradigm)
    if stats["total"] > 0:
        reg_acc = (stats["regex_pass"] / stats["total"]) * 100
        llm_acc = (stats["llm_pass"] / stats["total"]) * 100
    else:
        reg_acc, llm_acc = 0, 0
        
    regex_scores.append(reg_acc)
    llm_scores.append(llm_acc)
    
    # Print the CSV string for the user to copy
    print(f"{paradigm},{reg_acc:.2f}%,{llm_acc:.2f}%,{abs(reg_acc-llm_acc):.2f}%")

print("--------------------------------------------\n")

# --- PLOT: Comparative Bar Chart ---
sns.set_theme(style="whitegrid")
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
rects1 = ax.bar(x - width/2, regex_scores, width, label='Level 1: Regex Grader', color='#1f77b4')
rects2 = ax.bar(x + width/2, llm_scores, width, label='Level 2: LLM Judge', color='#ff7f0e')

ax.set_ylabel('Accuracy (%)', fontweight='bold')
ax.set_title('Dual-Level Evaluation: Strict Determinism vs. Semantic Judge', fontweight='bold', size=14)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right', fontweight='bold')
ax.legend()
ax.set_ylim(0, 110)

# Add value labels
for rects in [rects1, rects2]:
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.1f}%',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  
                    textcoords="offset points",
                    ha='center', va='bottom')

plt.tight_layout()
plt.show()